# Runtime

LangChain의 `create_agent`는 내부적으로 LangGraph의 런타임을 사용합니다. Runtime은 에이전트 실행 중 도구와 미들웨어에서 접근할 수 있는 컨텍스트 정보를 제공합니다.

**Runtime 구성 요소:**

| 구성 요소 | 설명 |
|:---|:---|
| **Context** | 사용자 ID, 데이터베이스 연결 등 정적 정보 |
| **Store** | 장기 메모리를 위한 `BaseStore` 인스턴스 |
| **Stream Writer** | `"custom"` 스트림 모드로 정보 스트리밍 |

런타임 정보는 도구와 미들웨어 내에서 `runtime` 매개변수를 통해 액세스할 수 있습니다.

> 참고 문서: [LangChain Runtime](https://docs.langchain.com/oss/python/langchain/runtime)

## 환경 설정

Runtime 튜토리얼을 시작하기 전에 필요한 환경을 설정합니다. `dotenv`를 사용하여 API 키를 로드하고, LangSmith 추적을 설정합니다.

아래 코드는 환경 변수를 로드하고 LangSmith 추적을 활성화합니다.

In [1]:
# 환경 변수 로드
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv(override=True)

# LangSmith 추적 설정
logging.langsmith("LangGraph-V1-Tutorial")

LangChain/LangSmith API Key가 설정되지 않았습니다. 참고: https://wikidocs.net/250954


---

## 도구에서 Runtime 액세스

도구 내에서 `ToolRuntime` 매개변수를 사용하여 `Runtime` 객체에 액세스할 수 있습니다. 이를 통해 다음 기능을 수행할 수 있습니다:

- **Context 접근**: 사용자 정보, 세션 데이터 등
- **Store 읽기/쓰기**: 장기 메모리 관리
- **Stream Writer**: 진행 상황 스트리밍

`ToolRuntime` 매개변수는 도구 시그니처에 추가하면 자동으로 주입되며, LLM에는 노출되지 않습니다.

다음 섹션에서 Context, Store, Stream Writer에 접근하는 구체적인 예시를 확인할 수 있습니다.

### Context 액세스

`create_agent`로 에이전트를 생성할 때 `context_schema`를 지정하여 에이전트 `Runtime`에 저장될 `context`의 구조를 정의할 수 있습니다. Context는 dataclass 또는 Pydantic 모델로 정의하며, 에이전트 호출 시 `context` 매개변수로 전달합니다.

도구에서는 `runtime.context`를 통해 Context 객체에 접근할 수 있습니다. `ToolRuntime[ContextType]` 형태로 타입 힌트를 지정하면 IDE에서 자동 완성을 지원받을 수 있습니다.

아래 코드는 도구에서 Context에 접근하여 사용자 정보를 활용하는 예시입니다.

In [2]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool, ToolRuntime
from dataclasses import dataclass
from langchain_teddynote.messages import invoke_graph


# 사용자 정보를 담는 Context 스키마
@dataclass
class UserContext:
    user_id: str
    user_name: str
    user_email: str


# 사용자 정보를 가져오는 도구
@tool
def get_user_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get information about the current user."""
    # Context에서 사용자 정보 가져오기
    user_id = runtime.context.user_id
    user_name = runtime.context.user_name
    user_email = runtime.context.user_email

    return f"User ID: {user_id}, Name: {user_name}, Email: {user_email}"


# 개인화된 인사를 생성하는 도구
@tool
def personalized_greeting(runtime: ToolRuntime[UserContext]) -> str:
    """Generate a personalized greeting for the user."""
    user_name = runtime.context.user_name
    return f"안녕하세요, {user_name}님! 무엇을 도와드릴까요?"

# LLM 초기화 (OpenAI 키 사용 시 gpt-5.2, gpt-4.1-mini 등으로 변경 가능)
model = init_chat_model("claude-sonnet-4-5")

# 에이전트 생성
agent = create_agent(
    model=model,
    tools=[personalized_greeting, get_user_info],
    context_schema=UserContext,
)

In [3]:
# Context를 전달하여 호출
invoke_graph(
    agent,
    inputs={"messages": [{"role": "user", "content": "안녕하세요? 반갑습니다."}]},
    context=UserContext(
        user_id="user_123", user_name="김철수", user_email="chulsoo@example.com"
    ),
)


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

[{'id': 'toolu_013YBxiagG2daDWXE9v2z9Hk', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'personalized_greeting', 'type': 'tool_use'}]
Tool Calls:
  personalized_greeting (toolu_013YBxiagG2daDWXE9v2z9Hk)
 Call ID: toolu_013YBxiagG2daDWXE9v2z9Hk
  Args:

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: personalized_greeting

안녕하세요, 김철수님! 무엇을 도와드릴까요?



🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

안녕하세요, 김철수님! 만나서 반갑습니다! 😊

오늘 무엇을 도와드릴까요?


### Store 액세스 (장기 메모리)

도구 내에서 `runtime.store`를 사용하여 장기 메모리에 액세스할 수 있습니다. Store는 대화 세션을 넘어서 데이터를 영구 저장하며, `get()`, `put()` 메서드로 데이터를 읽고 씁니다.

Store의 키는 네임스페이스(튜플)와 키(문자열)로 구성되어 데이터를 체계적으로 관리할 수 있습니다.

아래 코드는 Store를 사용하여 사용자 설정을 저장하고 조회하는 예시입니다.

In [4]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore


# 사용자 ID만 포함하는 간단한 Context
@dataclass
class Context:
    user_id: str


# 사용자 이메일 설정을 가져오는 도구
@tool
def fetch_user_email_preferences(runtime: ToolRuntime[Context]) -> str:
    """Fetch the user's email writing style preferences from the store."""
    user_id = runtime.context.user_id

    # 기본 설정
    preferences: str = "사용자는 간결하고 정중한 이메일 작성을 선호합니다."

    # Store에서 사용자 설정 가져오기
    if runtime.store:
        if memory := runtime.store.get(("users",), user_id):
            preferences = memory.value["preferences"]

    return preferences


# 사용자 설정을 저장하는 도구
@tool
def save_user_preference(preference: str, runtime: ToolRuntime[Context]) -> str:
    """Save user's preference settings to the store."""
    user_id = runtime.context.user_id

    if runtime.store:
        runtime.store.put(("users",), user_id, {"preferences": preference})
        return f"설정이 저장되었습니다: {preference}"

    return "Store를 사용할 수 없습니다."


# Store와 함께 에이전트 생성
store = InMemoryStore()

# 초기 데이터 설정 - 사용자별 이메일 작성 스타일 저장
store.put(
    ("users",),
    "user_123",
    {"preferences": "사용자는 상세하고 기술적인 설명을 선호합니다."},
)

# 에이전트 생성 (Store 전달)
agent = create_agent(
    model=model,
    tools=[fetch_user_email_preferences, save_user_preference],
    context_schema=Context,
    store=store,  # Store 전달
)

In [5]:
# Store에서 설정 가져오기
invoke_graph(
    agent,
    {"messages": [{"role": "user", "content": "제 이메일 작성 스타일 설정이 어떻게 되어 있나요?"}]},
    context=Context(user_id="user_123"),
)


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

[{'id': 'toolu_016HhPtqBCcWea7FrCrw8A2i', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'fetch_user_email_preferences', 'type': 'tool_use'}]
Tool Calls:
  fetch_user_email_preferences (toolu_016HhPtqBCcWea7FrCrw8A2i)
 Call ID: toolu_016HhPtqBCcWea7FrCrw8A2i
  Args:

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: fetch_user_email_preferences

사용자는 상세하고 기술적인 설명을 선호합니다.



🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

현재 회원님의 이메일 작성 스타일 설정은 **"상세하고 기술적인 설명을 선호"**로 되어 있습니다.

이 설정에 따르면 이메일을 작성할 때 구체적이고 세부적인 내용을 포함하며, 기술적인 용어나 전문적인 설명을 사용하는 스타일이 적용됩니다. 

다른 스타일로 변경을 원하시면 말씀해 주세요!


### Stream Writer 액세스

도구 내에서 `runtime.get_stream_writer()` 또는 `runtime.stream_writer`를 사용하여 커스텀 업데이트를 스트리밍할 수 있습니다. 이는 장시간 실행되는 작업에서 사용자에게 진행 상황을 실시간으로 알려줄 때 유용합니다.

스트리밍된 업데이트는 `stream_mode="custom"`으로 수신할 수 있습니다.

아래 코드는 Stream Writer를 사용하여 진행 상황을 스트리밍하는 예시입니다.

In [6]:
from langchain.tools import tool, ToolRuntime
import time


# 대용량 데이터 처리를 시뮬레이션하는 도구
@tool
def process_large_dataset(num_items: int, runtime: ToolRuntime) -> str:
    """Process a large dataset and report progress."""
    # Stream Writer 가져오기
    writer = runtime.stream_writer

    # 진행 상황 스트리밍
    for i in range(0, num_items, 10):
        progress = min(i + 10, num_items)
        writer({"stage": "processing", "progress": progress, "total": num_items})
        time.sleep(0.1)  # 작업 시뮬레이션

    # 완료 상태 스트리밍
    writer({"stage": "completed", "total": num_items})
    return f"Successfully processed {num_items} items!"


# 에이전트 생성
agent = create_agent(
    model=model,
    tools=[process_large_dataset],
)

# 커스텀 스트림 모드로 진행 상황 추적
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "Process 50 items"}]},
    stream_mode="custom",
):
    if "progress" in chunk:
        # 진행률 계산 및 출력
        percentage = (chunk["progress"] / chunk["total"]) * 100
        print(f"Progress: {percentage:.0f}%")
    elif "stage" in chunk and chunk["stage"] == "completed":
        # 완료 메시지 출력
        print(f"Completed processing {chunk['total']} items!")

Progress: 20%
Progress: 40%


Progress: 60%
Progress: 80%


Progress: 100%
Completed processing 50 items!


---

## 미들웨어에서 Runtime 액세스

미들웨어에서 `Runtime` 객체에 액세스하여 동적 프롬프트를 생성하거나, 메시지를 수정하거나, 사용자 컨텍스트에 따라 에이전트 동작을 제어할 수 있습니다. 미들웨어 함수의 `runtime` 매개변수를 통해 Context, Store 등에 접근합니다.

아래 섹션에서는 Dynamic Prompt와 Before/After Model 미들웨어에서 Runtime을 사용하는 방법을 살펴봅니다.

### Dynamic Prompt에서 Runtime 사용

`@dynamic_prompt` 데코레이터로 정의된 미들웨어에서 `request.runtime`을 통해 Context에 접근할 수 있습니다. 이를 활용하여 사용자별로 다른 시스템 프롬프트를 동적으로 생성할 수 있습니다.

아래 코드는 사용자 언어에 따라 다른 시스템 프롬프트를 생성하는 예시입니다.

In [7]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain.tools import tool


# 사용자 이름과 언어를 담는 Context 스키마
@dataclass
class Context:
    user_name: str
    language: str


# 날씨 정보를 가져오는 간단한 도구
@tool
def get_weather(city: str) -> str:
    """Get the weather in a city."""
    return f"The weather in {city} is sunny!"


# 동적 시스템 프롬프트 미들웨어
@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    # Runtime에서 Context 가져오기
    user_name = request.runtime.context.user_name
    language = request.runtime.context.language

    # 사용자 언어에 따라 다른 프롬프트
    if language == "Korean":
        system_prompt = f"You are a helpful assistant. Address the user as '{user_name}'. Always respond in Korean."
    else:
        system_prompt = f"You are a helpful assistant. Address the user as '{user_name}'. Always respond in English."

    return system_prompt


# 미들웨어를 포함한 에이전트 생성
agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[dynamic_system_prompt],
    context_schema=Context,
)

# 한국어 사용자로 호출
print("=== Korean User ===")
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    context=Context(user_name="김철수", language="Korean"),
)
print(result["messages"][-1].content)

# 영어 사용자로 호출
print("\n=== English User ===")
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    context=Context(user_name="John", language="English"),
)
print(result["messages"][-1].content)

=== Korean User ===


김철수님, SF의 날씨는 맑습니다!

=== English User ===


John, the weather in San Francisco is sunny!


### Before/After Model에서 Runtime 사용

`@before_model`과 `@after_model` 데코레이터로 정의된 미들웨어에서도 `runtime` 매개변수를 통해 Context에 접근할 수 있습니다. 이를 활용하여 모델 호출 전후에 로깅, 검증, 변환 등의 작업을 수행할 수 있습니다.

아래 코드는 모델 호출 전후에 사용자 정보를 로깅하는 예시입니다.

In [8]:
from langchain.agents import AgentState
from langchain.agents.middleware import before_model, after_model
from langgraph.runtime import Runtime
from dataclasses import dataclass


# 세션 정보를 담는 Context 스키마
@dataclass
class Context:
    user_name: str
    session_id: str


# 모델 호출 전 로깅 미들웨어
@before_model
def log_before_model(state: AgentState, runtime: Runtime[Context]) -> dict | None:
    """모델 호출 전 로깅"""
    print(
        f"[Before Model] User: {runtime.context.user_name}, Session: {runtime.context.session_id}"
    )
    print(f"[Before Model] Messages count: {len(state['messages'])}")
    return None


# 모델 호출 후 로깅 미들웨어
@after_model
def log_after_model(state: AgentState, runtime: Runtime[Context]) -> dict | None:
    """모델 호출 후 로깅"""
    print(f"[After Model] User: {runtime.context.user_name}")
    print(f"[After Model] Response generated for session: {runtime.context.session_id}")
    return None


# 미들웨어를 포함한 에이전트 생성
agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[log_before_model, log_after_model],
    context_schema=Context,
)

# 에이전트 호출
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in Seoul?"}]},
    context=Context(user_name="John Smith", session_id="session_456"),
)

print(f"\nFinal response: {result['messages'][-1].content}")

[Before Model] User: John Smith, Session: session_456
[Before Model] Messages count: 1


[After Model] User: John Smith
[After Model] Response generated for session: session_456
[Before Model] User: John Smith, Session: session_456
[Before Model] Messages count: 3


[After Model] User: John Smith
[After Model] Response generated for session: session_456

Final response: The weather in Seoul is sunny!


---

## 종합 예제: 사용자 컨텍스트 기반 에이전트

Runtime의 모든 기능을 활용한 실용적인 예제입니다. Context로 사용자 정보를 전달하고, Store로 검색 기록을 관리하며, 미들웨어로 동적 프롬프트와 사용량 추적을 구현합니다.

아래 코드는 사용자 등급에 따라 다른 기능을 제공하는 에이전트 예시입니다.

In [9]:
from dataclasses import dataclass
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import dynamic_prompt, ModelRequest, before_model
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore
from langgraph.runtime import Runtime


# 사용자 정보를 담는 Context 스키마
@dataclass
class UserContext:
    user_id: str
    user_name: str
    user_tier: str  # "free", "premium", "enterprise"
    language: str  # "ko", "en"


# 데이터베이스 검색 도구
@tool
def search_database(query: str, runtime: ToolRuntime[UserContext]) -> str:
    """Search the database. Access level depends on user tier."""
    user_tier = runtime.context.user_tier

    # 사용자 등급에 따라 다른 결과 제공
    if user_tier == "enterprise":
        return f"Full database search results for: {query} (Enterprise access)"
    elif user_tier == "premium":
        return f"Premium search results for: {query}"
    else:
        return f"Basic search results for: {query} (Limited to 10 results)"


# 사용자 검색 기록을 가져오는 도구
@tool
def get_user_history(runtime: ToolRuntime[UserContext]) -> str:
    """Get user's search history from store."""
    user_id = runtime.context.user_id

    if runtime.store:
        if history := runtime.store.get(("history",), user_id):
            return f"Recent searches: {history.value['searches']}"

    return "No search history found"


# 검색 기록을 저장하는 도구
@tool
def save_search(query: str, runtime: ToolRuntime[UserContext]) -> str:
    """Save search query to user history."""
    user_id = runtime.context.user_id

    if runtime.store:
        # 기존 히스토리 가져오기
        existing = runtime.store.get(("history",), user_id)
        searches = existing.value["searches"] if existing else []

        # 새 검색어 추가 (최근 5개만 유지)
        searches.append(query)
        runtime.store.put(("history",), user_id, {"searches": searches[-5:]})

        return f"Saved search: {query}"

    return "Store not available"


# 동적 프롬프트 - 사용자 언어에 따라 변경
@dynamic_prompt
def multilingual_prompt(request: ModelRequest) -> str:
    user_name = request.runtime.context.user_name
    language = request.runtime.context.language
    user_tier = request.runtime.context.user_tier

    if language == "ko":
        prompt = f"당신은 도움이 되는 어시스턴트입니다. 사용자를 '{user_name}'님으로 호칭하세요."
        if user_tier == "enterprise":
            prompt += (
                " 이 사용자는 엔터프라이즈 회원이므로 모든 기능에 액세스할 수 있습니다."
            )
    else:
        prompt = f"You are a helpful assistant. Address the user as {user_name}."
        if user_tier == "enterprise":
            prompt += " This is an enterprise user with full access."

    return prompt


# 사용량 추적 미들웨어
@before_model
def track_usage(state: AgentState, runtime: Runtime[UserContext]) -> dict | None:
    """Track API usage for billing"""
    user_id = runtime.context.user_id
    user_tier = runtime.context.user_tier

    print(f"[Usage Tracker] User: {user_id}, Tier: {user_tier}")

    # 무료 사용자의 경우 사용량 제한 확인
    if user_tier == "free":
        if runtime.store:
            usage = runtime.store.get(("usage",), user_id)
            count = usage.value["count"] if usage else 0

            if count >= 10:
                print("[Usage Tracker] Free tier limit reached!")
                # 실제로는 여기서 실행을 중단할 수 있음

            # 사용량 업데이트
            runtime.store.put(("usage",), user_id, {"count": count + 1})

    return None


# Store 생성 및 초기 데이터 설정
store = InMemoryStore()
store.put(
    ("history",), "user_001", {"searches": ["Python tutorial", "LangChain guide"]}
)
store.put(("usage",), "user_002", {"count": 5})

# 에이전트 생성
agent = create_agent(
    model=model,
    tools=[search_database, get_user_history, save_search],
    middleware=[multilingual_prompt, track_usage],
    context_schema=UserContext,
    store=store,
)

# 테스트 1: 엔터프라이즈 사용자 (한국어)
print("=== Test 1: Enterprise User (Korean) ===")
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Search for 'machine learning'"}]},
    context=UserContext(
        user_id="user_001", user_name="김철수", user_tier="enterprise", language="ko"
    ),
)
print(result["messages"][-1].content)

# 테스트 2: 무료 사용자 (영어)
print("\n=== Test 2: Free User (English) ===")
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Search for 'data science'"}]},
    context=UserContext(
        user_id="user_002", user_name="John Doe", user_tier="free", language="en"
    ),
)
print(result["messages"][-1].content)

# 테스트 3: 검색 기록 조회
print("\n=== Test 3: Check Search History ===")
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my search history?"}]},
    context=UserContext(
        user_id="user_001", user_name="김철수", user_tier="enterprise", language="ko"
    ),
)
print(result["messages"][-1].content)

=== Test 1: Enterprise User (Korean) ===
[Usage Tracker] User: user_001, Tier: enterprise


[Usage Tracker] User: user_001, Tier: enterprise


김철수님, 'machine learning'에 대한 검색을 완료했습니다. 엔터프라이즈 회원이시므로 전체 데이터베이스 검색 결과를 확인하실 수 있습니다. 검색 기록도 자동으로 저장되었습니다.

=== Test 2: Free User (English) ===
[Usage Tracker] User: user_002, Tier: free


[Usage Tracker] User: user_002, Tier: free


I've searched for "data science" for you, John Doe. The search returned basic results (limited to 10 results based on your access level). Your search has also been saved to your history for future reference.

=== Test 3: Check Search History ===
[Usage Tracker] User: user_001, Tier: enterprise


[Usage Tracker] User: user_001, Tier: enterprise


김철수님, 최근 검색 기록은 다음과 같습니다:

1. Python tutorial
2. LangChain guide
3. machine learning

엔터프라이즈 회원이신 김철수님께서는 모든 데이터베이스 검색 기능을 제한 없이 이용하실 수 있습니다. 더 궁금하신 사항이 있으시면 말씀해 주세요!


---

## 실전 패턴

### 데이터베이스 연결 전달

Context를 사용하여 데이터베이스 연결 객체를 도구에 전달할 수 있습니다. 이 패턴을 사용하면 도구에서 직접 데이터베이스에 접근하여 쿼리를 실행할 수 있습니다.

아래 코드는 데이터베이스 연결을 Context로 전달하는 예시입니다.

In [10]:
from dataclasses import dataclass
from typing import Any


# 데이터베이스 연결을 담는 Context 스키마
@dataclass
class DatabaseContext:
    db_connection: Any  # 실제로는 데이터베이스 연결 객체
    user_id: str


# 데이터베이스 쿼리를 실행하는 도구
@tool
def query_database(sql: str, runtime: ToolRuntime[DatabaseContext]) -> str:
    """Execute SQL query on the database."""
    db = runtime.context.db_connection
    user_id = runtime.context.user_id

    # 실제 데이터베이스 쿼리 실행
    # result = db.execute(sql)

    return f"Query executed for user {user_id}: {sql}"


# 사용 예시 (실제 DB 연결 대신 None 사용)
# agent = create_agent(
#     model=model,
#     tools=[query_database],
#     context_schema=DatabaseContext
# )
#
# result = agent.invoke(
#     {"messages": [{"role": "user", "content": "Query user data"}]},
#     context=DatabaseContext(db_connection=db, user_id="user_123")
# )

### 인증 및 권한 검사

미들웨어에서 Context를 사용하여 사용자 인증 및 권한을 검사할 수 있습니다. `@before_agent` 미들웨어에서 권한이 없는 요청을 사전에 차단하면 보안을 강화할 수 있습니다.

아래 코드는 사용자 권한을 검사하는 미들웨어 예시입니다.

In [11]:
from langchain.agents.middleware import before_agent
from langgraph.runtime import Runtime
from typing import Any


# 인증 정보를 담는 Context 스키마
@dataclass
class AuthContext:
    user_id: str
    permissions: list[str]


# 권한 검사 미들웨어
@before_agent(can_jump_to=["end"])
def check_permissions(
    state: AgentState, runtime: Runtime[AuthContext]
) -> dict[str, Any] | None:
    """Check if user has required permissions"""
    permissions = runtime.context.permissions

    # 메시지에서 요청된 작업 확인
    if state["messages"]:
        content = state["messages"][0].content.lower()

        # 관리자 작업 요청 시 권한 확인
        if "delete" in content or "remove" in content:
            if "admin" not in permissions:
                return {
                    "messages": [
                        {
                            "role": "assistant",
                            "content": "You don't have permission to perform this action.",
                        }
                    ],
                    "jump_to": "end",
                }

    return None


# 사용 예시
agent = create_agent(
    model=model,
    tools=[],
    middleware=[check_permissions],
    context_schema=AuthContext
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Delete user data"}]},
    context=AuthContext(user_id="user_123", permissions=["user", "read", "write"])
)

print(result["messages"][-1].content)

You don't have permission to perform this action.


### 요청별 설정

각 요청에 대한 특정 설정을 Context를 통해 전달할 수 있습니다. 타임아웃, 토큰 제한, 로깅 수준 등 요청마다 다른 설정이 필요한 경우 유용합니다.

아래 코드는 요청별 설정을 Context로 전달하는 예시입니다.

In [12]:
# 요청별 설정을 담는 Context 스키마
@dataclass
class RequestContext:
    user_id: str
    verbose: bool
    timeout: int
    max_tokens: int


# 요청 설정에 따라 처리하는 도구
@tool
def process_request(query: str, runtime: ToolRuntime[RequestContext]) -> str:
    """Process request with custom settings."""
    verbose = runtime.context.verbose
    timeout = runtime.context.timeout

    if verbose:
        print(f"Processing with timeout: {timeout}s")

    # 설정에 따라 처리
    return f"Processed: {query}"


# 에이전트 생성
agent = create_agent(
    model=model, tools=[process_request], context_schema=RequestContext
)

# 요청별로 다른 설정 사용
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Process my request"}]},
    context=RequestContext(
        user_id="user_123", verbose=True, timeout=30, max_tokens=1000
    ),
)

print(result["messages"][-1].content)

I'd be happy to help process your request! However, I need to know what specific query or request you'd like me to process. Could you please provide the details of what you need help with?


---

## 정리

이 튜토리얼에서는 LangGraph 에이전트의 Runtime 기능을 학습했습니다.

**핵심 개념 요약:**

| 개념 | 설명 | 접근 방법 |
|:---|:---|:---|
| **Context** | 사용자 정보, 세션 데이터 등 정적 정보 | `runtime.context` |
| **Store** | 대화 세션을 넘어선 장기 메모리 | `runtime.store.get()`, `put()` |
| **Stream Writer** | 커스텀 진행 상황 스트리밍 | `runtime.stream_writer` |

**실전 패턴:**
- 데이터베이스 연결을 Context로 전달하여 도구에서 직접 쿼리 실행
- 미들웨어에서 사용자 권한 검사로 보안 강화
- 요청별 설정(타임아웃, 토큰 제한 등)을 Context로 전달

**다음 단계:**
- 구조화된 출력(Structured Output)을 사용한 에이전트 구축 학습